# LightGBM + GraphSAGE joint-decision experiment
Frozen models; GraphSAGE scores ML >= 0.10, including ML >= 0.96. Band-specific graph thresholds are chosen on full validation only. Full test compares the selected policy with the previous cascade and exports all final FP/FN.


In [ ]:
from pathlib import Path
import json, os, sys, tarfile

MODEL_NAME = 'lgbm'
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
OUTPUT = WORK / 'cascade_run_lgbm_joint_policy_v1'
WORK.mkdir(parents=True, exist_ok=True)

# Kaggle may expose the dataset as extracted files or as a single archive.
bundle = INPUT / 'datasets/nhatnguyen2101/optc-cascade-bundle'
required = ('bundle_manifest.json', 'provenance/graph.sqlite', 'provenance/model_numpy.npz',
            'code/cascade_runtime.py', f'models/{MODEL_NAME}_metrics.json')
if not all((bundle / name).is_file() for name in required):
    extracted = [p.parent for p in INPUT.glob('**/bundle_manifest.json')
                 if all((p.parent / name).is_file() for name in required)]
    if len(extracted) == 1:
        bundle = extracted[0]
    else:
        archives = sorted(INPUT.glob('**/provenance-cascade-bundle-v3.tar.gz'))
        if len(archives) != 1:
            raise RuntimeError(f'Expected one extracted v3 bundle or archive; found {len(extracted)} extracted, {len(archives)} archives')
        bundle = WORK / 'cascade_bundle_v3'
        bundle.mkdir(exist_ok=True)
        with tarfile.open(archives[0], 'r:gz') as tar:
            for member in tar.getmembers():
                target = (bundle / member.name).resolve()
                if not target.is_relative_to(bundle.resolve()) or not member.isfile():
                    raise ValueError(f'Unsafe bundle member: {member.name}')
            tar.extractall(bundle)
if json.loads((bundle / 'bundle_manifest.json').read_text()).get('schema') != 3:
    raise RuntimeError('RF, XGB and LGBM notebooks require the v3 cascade bundle')
sys.path.insert(0, str(bundle / 'code'))
import cascade_runtime as cr
import numpy as np

# Notebook-only experiment. The packaged RF/XGB/LGBM runtime is unchanged.
# Score GraphSAGE for every event with ML >= 0.10, including ML >= 0.96.
# The below-0.10 segment keeps the frozen ML baseline decision.
cr.LOW = (0.10,)
cr.HIGH = (1.01,)
LOW_GRAPH = (0.95, 0.98, 0.995)
MID_GRAPH = (0.85, 0.90, 0.95, 0.98)
HIGH_GRAPH = (0.70, 0.85, 0.95, 0.98)

def joint_policies(threshold):
    # Validation-only grid; every candidate sends ML >= 0.10 to GraphSAGE.
    return [dict(mode='baseline', threshold=threshold)] + [dict(mode='cascade', low=0.10, high=1.01,
                 low_graph=lo, middle_graph=mid, high_graph=hi,
                 baseline_threshold=threshold)
            for lo in LOW_GRAPH for mid in MID_GRAPH for hi in HIGH_GRAPH]

def joint_predictions(ml, graph, policy):
    if policy['mode'] == 'baseline':
        return ml >= policy['threshold']
    finite = np.isfinite(graph)
    pred = ml >= policy['baseline_threshold']
    boundaries = ((0.10, 0.30, 'low_graph'),
                  (0.30, 0.96, 'middle_graph'),
                  (0.96, 1.01, 'high_graph'))
    for lower, upper, key in boundaries:
        band = (ml >= lower) & (ml < upper) & finite
        pred[band] = graph[band] >= policy[key]
    # A missing graph lookup retains the frozen ML decision on both splits.
    return pred

def choose_joint(options, totals):
    # Baseline at index zero is an integrity check, not an eligible new policy.
    return options[max(range(1, len(options)), key=lambda i: cr.metrics(totals[i])['f1'])]

cr.policies = joint_policies
cr.predictions = joint_predictions
cr.choose = choose_joint
run = cr.run

host_root = INPUT / 'datasets/nhatnguyen2101/optc-host-ml-full-input'
manifest = host_root / 'manifest.json'
if not (manifest.is_file() and (host_root / 'assets/vocab.json').is_file()):
    candidates = [p for p in INPUT.glob('**/manifest.json')
                  if (p.parent / 'assets/vocab.json').is_file()]
    if len(candidates) == 1:
        manifest = candidates[0]
    else:
        archives = sorted(INPUT.glob('**/host-input.tar'))
        if len(archives) != 1:
            raise RuntimeError(f'Expected one extracted Host dataset or archive; found {len(candidates)} extracted, {len(archives)} archives')
        host_root = WORK / 'host_shards'
        host_root.mkdir(exist_ok=True)
        with tarfile.open(archives[0], 'r') as tar:
            for member in tar.getmembers():
                target = (host_root / member.name).resolve()
                if not target.is_relative_to(host_root.resolve()) or not member.isfile():
                    raise ValueError(f'Unsafe Host archive member: {member.name}')
            tar.extractall(host_root)
        manifest = host_root / 'manifest.json'

print('Host model:', MODEL_NAME)
print('Cascade bundle:', bundle)
print('Host shards:', manifest)
print('Output:', OUTPUT)
result = run(manifest, bundle, OUTPUT, model_name=MODEL_NAME)
print('Results ZIP:', OUTPUT / 'optc-two-stage-cascade-results.zip')


In [ ]:
# Audit the frozen validation-selected policy against the previous measured cascade.
# This cell reads TEST labels only after cr.run has selected and frozen its policy.
from collections import Counter
import zipfile
from cascade_runtime import counts, metrics, atomic_json

OLD_POLICY = dict(mode='cascade', low=0.30, high=0.96, graph=0.95,
                  baseline_threshold=json.loads((bundle / 'models/lgbm_metrics.json').read_text())['validation']['threshold'])
selected = json.loads((OUTPUT / 'validation_calibration.json').read_text())['selected']
assert selected['low'] == 0.10 and selected['high'] == 1.01
shards = json.loads(manifest.read_text())['shards']
old_counts = np.zeros(4, dtype=np.int64)
new_counts = np.zeros(4, dtype=np.int64)
transition = Counter()
old_new_fp, old_new_fn = Counter(), Counter()
for i, shard in enumerate(shards):
    if shard['split'] != 'test':
        continue
    with np.load(OUTPUT / f'scores-{i:04d}.npz', allow_pickle=False) as d:
        y, ml, graph = d['y'], d['ml'], d['graph']
        old = ml >= OLD_POLICY['high']
        old_band = (ml >= OLD_POLICY['low']) & (ml < OLD_POLICY['high'])
        old[old_band] = np.where(np.isfinite(graph[old_band]),
                                 graph[old_band] >= OLD_POLICY['graph'],
                                 ml[old_band] >= OLD_POLICY['baseline_threshold'])
        new = joint_predictions(ml, graph, selected)
        baseline = ml >= OLD_POLICY['baseline_threshold']
        old_counts += counts(y, old)
        new_counts += counts(y, new)
        for actual, old_value, new_value, base_value in zip(y, old, new, baseline):
            transition[f'{"P" if actual else "N"}:{int(old_value)}->{int(new_value)}'] += 1
            if actual == 0 and not base_value:
                old_new_fp['old'] += int(old_value)
                old_new_fp['new'] += int(new_value)
                old_new_fp['persist'] += int(old_value and new_value)
            if actual == 1 and base_value:
                old_new_fn['old'] += int(not old_value)
                old_new_fn['new'] += int(not new_value)
                old_new_fn['persist'] += int((not old_value) and (not new_value))
old_report, new_report = metrics(old_counts), metrics(new_counts)
measured = json.loads((OUTPUT / 'two_stage_cascade_metrics.json').read_text())
assert new_report['confusion'] == measured['cascade']['confusion']
assert old_report['confusion'] == {'tp': 39654, 'fp': 568, 'fn': 656, 'tn': 1208811}, 'Old cascade did not reproduce'
assert (old_new_fp['old'], old_new_fn['old']) == (161, 31), 'Old new-error cohorts did not reproduce'
comparison = dict(selection_split='val', evaluation_split='test',
                  old_policy=OLD_POLICY, selected_policy=selected,
                  old_cascade=old_report, new_cascade=new_report,
                  transition=dict(transition),
                  new_errors_vs_frozen_ml=dict(fp=dict(old=old_new_fp['old'], persist=old_new_fp['persist'], corrected=old_new_fp['old']-old_new_fp['persist'], new_total=old_new_fp['new']),
                                               fn=dict(old=old_new_fn['old'], persist=old_new_fn['persist'], corrected=old_new_fn['old']-old_new_fn['persist'], new_total=old_new_fn['new'])))
atomic_json(OUTPUT / 'old_vs_joint_policy.json', comparison)
with zipfile.ZipFile(OUTPUT / 'optc-two-stage-cascade-results.zip', 'a', zipfile.ZIP_DEFLATED) as archive:
    archive.write(OUTPUT / 'old_vs_joint_policy.json', arcname='old_vs_joint_policy.json')
print(json.dumps(comparison, indent=2))


In [ ]:
# Measured validation/test diagnostics, after the full cascade finishes.
import matplotlib.pyplot as plt
import numpy as np
import zipfile
from IPython.display import display, Image

cal = json.loads((OUTPUT / 'validation_calibration.json').read_text())
measured = json.loads((OUTPUT / 'two_stage_cascade_metrics.json').read_text())
base, fused = measured['baseline'], measured['cascade']
previous = json.loads((OUTPUT / 'old_vs_joint_policy.json').read_text())['old_cascade']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, title, report in zip(axes, ('Frozen Host ML', 'Previous cascade', 'Joint policy'), (base, previous, fused)):
    c = report['confusion']
    matrix = np.array([[c['tn'], c['fp']], [c['fn'], c['tp']]])
    ax.imshow(np.log1p(matrix), cmap='Blues')
    for row in range(2):
        for col in range(2):
            ax.text(col, row, f'{matrix[row, col]:,}', ha='center', va='center',
                    bbox={'boxstyle':'round', 'facecolor':'white', 'alpha':0.85})
    ax.set(xlabel='Predicted', ylabel='Actual', title=title,
           xticks=[0, 1], yticks=[0, 1], xticklabels=['Benign', 'Malicious'],
           yticklabels=['Benign', 'Malicious'])
fig.suptitle(f'{MODEL_NAME.upper()} + GraphSAGE: full test confusion (log color scale)')
fig.tight_layout()
confusion_path = OUTPUT / 'confusion_matrix_comparison.png'
fig.savefig(confusion_path, dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

fig, (ax, route_ax) = plt.subplots(1, 2, figsize=(12, 4.5))
labels = ['Precision', 'Recall', 'F1']
base_values = [base['precision'], base['recall'], base['f1']]
old_values = [previous['precision'], previous['recall'], previous['f1']]
fused_values = [fused['precision'], fused['recall'], fused['f1']]
positions = np.arange(3)
ax.bar(positions - 0.24, base_values, width=0.24, label='Host ML')
ax.bar(positions, old_values, width=0.24, label='Previous cascade')
ax.bar(positions + 0.24, fused_values, width=0.24, label='Joint policy')
ax.set(xticks=positions, xticklabels=labels, ylim=(0, 1.05), ylabel='Test score', title='Full test metrics')
ax.legend()
route_ax.bar(['Direct', 'Graph'], [fused['rows'] - measured['routed'], measured['routed']])
route_ax.set(ylabel='Test events', title='Routing by validation-selected policy')
fig.tight_layout()
comparison_path = OUTPUT / 'metric_and_routing_comparison.png'
fig.savefig(comparison_path, dpi=180, bbox_inches='tight')
display(fig)
plt.close(fig)

print('Validation policy:', cal['selected'])
print(f"Validation F1: {next(x['f1'] for x in cal['grid'] if x['policy'] == cal['selected']):.4%}")
print(f"Test F1: joint {fused['f1']:.4%}; old cascade {previous['f1']:.4%}; baseline {base['f1']:.4%}")
print(f"Routed to GraphSAGE: {measured['routed']:,} / {fused['rows']:,}")
print(f"Missing graph lookups: {measured['missing_lookup']:,}")
source_index_path = OUTPUT / 'source_index.json'
source_index_path.write_text(json.dumps({s['source_id']: s['source']['path'] for s in json.loads(manifest.read_text())['sources']}, indent=2))
with zipfile.ZipFile(OUTPUT / 'optc-two-stage-cascade-results.zip', 'a', zipfile.ZIP_DEFLATED) as archive:
    archive.write(confusion_path, arcname=confusion_path.name)
    archive.write(comparison_path, arcname=comparison_path.name)
    archive.write(source_index_path, arcname=source_index_path.name)
print('Saved plots to', OUTPUT)
